# Self Attention 的公式

$$
\operatorname{SelfAttention}(X) = \operatorname{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

$$
Q = W_QX, \qquad K = W_KX, \qquad V = W_VX
$$

其中 $Q$、$K$、$V$ 分别由不同的权重矩阵 $W_Q$、$W_K$、$W_V$ 对输入 $X$ 进行线性变换得到。

### 1: 简化版本

$$
\operatorname{SelfAttention}(X) = \operatorname{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

In [1]:
import math
import torch
import torch.nn as nn

import warnings
# warnings.filterwarnings(action='ignore')

In [2]:

class SelfAttV1(nn.Module):
    def __init__(self, hidden_dim):
        super(SelfAttV1, self).__init__()
        self.hidden_dim = hidden_dim
        # 一般 Linear 都是默认有 bias
        # 一般来说， input dim 的 hidden dim
        self.query_proj = nn.Linear(hidden_dim, hidden_dim)
        self.key_proj = nn.Linear(hidden_dim, hidden_dim)
        self.value_proj = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x):
        # X shape is: (batch, seq_len, hidden_dim)， 一般是和 hidden_dim 相同
        # 但是 X 的 final dim 可以和 hidden_dim 不同
        Q = self.query_proj(x) # (batch, seq_len, hidden_dim)
        K = self.key_proj(x)   # (batch, seq_len, hidden_dim)
        V = self.value_proj(x) # (batch, seq_len, hidden_dim)

        # shape is: (batch, seq_len, seq_len)
        # torch.matmul 可以改成 Q @ K.transpose(-1, -2)
        # 其中 K 需要改成 shape 为： (batch, hidden_dim, seq_len)
        attention_value = torch.matmul(Q, K.transpose(-1, -2)) # dimension of attention_value is: (batch, seq_len, seq_len)
        attention_weights = torch.softmax(attention_value / math.sqrt(self.hidden_dim), dim = -1 )

        # print(attention_weights)
        # shape is: (batch, seq_len, hidden_dim)
        output = torch.matmul(attention_weights, V)
        return output

X = torch.rand(3, 2, 4)
net = SelfAttV1(4)
net(X)

tensor([[[-0.7338,  0.3274, -0.0993,  0.2617],
         [-0.7360,  0.3289, -0.0984,  0.2649]],

        [[-0.7343,  0.3436, -0.2023,  0.1619],
         [-0.7344,  0.3432, -0.2070,  0.1604]],

        [[-0.7302,  0.4211, -0.2368,  0.2327],
         [-0.7325,  0.4214, -0.2376,  0.2323]]], grad_fn=<UnsafeViewBackward0>)

### 2: Efficient implementation of self-attention using einsum

上面那哪些操作可以合并矩阵优化呢？- QKV 矩阵计算的时候，可以合并成一个大矩阵计算。
但是当前 transformers 实现中，其实是三个不同的 Linear 层


In [3]:
class SelfAttV2(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        # 这样可以进行加速, 那么为什么现在 Llama, qwen, gpt 等
        self.proj = nn.Linear(hidden_dim, hidden_dim * 3)

        # self.output_proj = nn.Linear(hidden_dim, hidden_dim) 这个单头是可以删除的，如果是多头的话，保留，因为保证是不同的头

    def forward(self, x):
        # X shape is: (batch, seq_len, hidden_dim)， 一般是和 hidden_dim 相同
        # 但是 X 的 final dim 可以和 hidden_dim 不同
        QKV = self.proj(x) # (batch, seq_len, hidden_dim * 3)
        
        # reshape 从希望的 q, k, 的形式
        Q, K, V = torch.split(QKV, self.hidden_dim, dim=-1)

        # print(x)

        att_weights = torch.softmax( 
            Q @ K.transpose(-1, -2) / math.sqrt(self.hidden_dim),
            dim=-1
        )
        output = att_weights @ V

        # return self.output_proj(output)

        return output

X = torch.rand(3, 2, 4)
net = SelfAttV2(4)

net(X)

tensor([[[ 0.3596, -0.5182,  0.3815,  0.8642],
         [ 0.3581, -0.5177,  0.3832,  0.8657]],

        [[ 0.4186, -0.5453,  0.4632,  1.0138],
         [ 0.4163, -0.5446,  0.4638,  1.0137]],

        [[ 0.1244, -0.4808,  0.3984,  0.8815],
         [ 0.1186, -0.4787,  0.3991,  0.8771]]], grad_fn=<UnsafeViewBackward0>)

### 3: 加入一些细节

In [4]:
# 1. dropout 位置
# 2. Attention mask
# 3. Output_proj 矩阵映射（可选）

class SelfAttV3(nn.Module):
    def __init__(self, hidden_dim, dropout_rate=0.1, *args, **kwargs):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.proj = nn.Linear(hidden_dim, hidden_dim * 3)
        self.attention_dropout = nn.Dropout(dropout_rate)

        # 可选 
        self.output_proj = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x, attention_mask=None):
        # X shape is: (batch, seq_len, hidden_dim)， 一般是和 hidden_dim 相同
        # 但是 X 的 final dim 可以和 hidden_dim 不同
        QKV = self.proj(x) # (batch, seq_len, hidden_dim * 3)
        
        # reshape 从希望的 q, k, 的形式
        Q, K, V = torch.split(QKV, self.hidden_dim, dim=-1)


        # (batch, seq_len, seq_len)
        att_weights = Q @ K.transpose(-1, -2) / math.sqrt(self.hidden_dim)
        if attention_mask is not None:
            att_weights = att_weights.masked_fill(attention_mask == 0, float('-inf'))
            
        print(att_weights)
        att_weights = torch.softmax(
                att_weights, 
                dim=-1
            )

        print(att_weights)

        # 这里在 BERT中的官方代码也说很奇怪，但是原文中这么用了，所以继承了下来
        # （用于 output 后面会更符合直觉）It is a bit strange that the dropout is applied to the attention weights
        #  rather than the output of the attention layer, 
        # but this is what was done in the original Transformer paper.
        att_weights = self.attention_dropout(att_weights)

        att_results = att_weights @ V
        output =  self.output_proj(att_results)
        
        return output

In [5]:
X = torch.rand(3, 4, 2)

# (batch, seq_len, seq_len)
mask = torch.tensor(
    [
        [1, 1, 1, 0],
        [1, 1, 0, 0],
        [1, 0, 0, 0],
    ]
)

mask = mask.unsqueeze(dim=1).repeat(1, 4, 1)
net = SelfAttV3(2)
net(X, mask)


tensor([[[ 2.2688e-02,  1.1646e-01,  2.5961e-02,        -inf],
         [ 7.9931e-02,  2.5584e-01,  7.7176e-02,        -inf],
         [ 6.1017e-02,  1.6511e-01,  5.6122e-02,        -inf],
         [ 8.3498e-02,  2.2927e-01,  7.7106e-02,        -inf]],

        [[ 2.7723e-01,  1.0739e-01,        -inf,        -inf],
         [ 1.1836e-01,  2.2369e-02,        -inf,        -inf],
         [ 1.4775e-01,  2.8242e-02,        -inf,        -inf],
         [ 2.5475e-01,  1.4644e-01,        -inf,        -inf]],

        [[-2.1625e-04,        -inf,        -inf,        -inf],
         [ 1.1505e-01,        -inf,        -inf,        -inf],
         [ 7.9850e-02,        -inf,        -inf,        -inf],
         [ 1.1027e-01,        -inf,        -inf,        -inf]]],
       grad_fn=<MaskedFillBackward0>)
tensor([[[0.3224, 0.3541, 0.3235, 0.0000],
         [0.3135, 0.3738, 0.3127, 0.0000],
         [0.3221, 0.3574, 0.3205, 0.0000],
         [0.3174, 0.3672, 0.3154, 0.0000]],

        [[0.5424, 0.4576, 

tensor([[[ 0.5642,  0.0222],
         [ 0.6230,  0.0205],
         [ 0.6256,  0.0198],
         [ 0.6240,  0.0202]],

        [[ 0.5880,  0.0224],
         [ 0.5909,  0.0211],
         [ 0.5900,  0.0215],
         [ 0.5790, -0.0088]],

        [[ 0.6445, -0.0358],
         [ 0.6445, -0.0358],
         [ 0.6445, -0.0358],
         [ 0.6445, -0.0358]]], grad_fn=<UnsafeViewBackward0>)

### 4. Final completed vsersion of self-attension

In [6]:
class SelfAttV4(nn.Module):
    def __init__(self, hidden_dim, dropout_rate=0.1):
        super().__init__()
        self.hidden_dim = hidden_dim

        self.query_proj = nn.Linear(hidden_dim, hidden_dim)
        self.key_proj = nn.Linear(hidden_dim, hidden_dim)
        self.value_proj = nn.Linear(hidden_dim, hidden_dim)

        # 一般是 0.1 的 dropout，一般写作 config.attention_probs_dropout_prob
        # hidden_dropout_prob 一般也是 0.1
        self.attention_dropout = nn.Dropout(dropout_rate)

        # 可选 
        # 这是 MultiHeadAttention 中的产物，这个留给 MultiHeadAttention 也没有问题；
        self.output_proj = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x, attention_mask=None):
        # X shape is: (batch, seq_len, hidden_dim)， 一般是和 hidden_dim 相同

        Q = self.query_proj(x) # (batch, seq_len, hidden_dim)
        K = self.key_proj(x)   # (batch, seq_len, hidden_dim)
        V = self.value_proj(x) # (batch, seq_len, hidden_dim)
        

    
        # (batch, seq_len, seq_len)
        att_weights = Q @ K.transpose(-1, -2) / math.sqrt(self.hidden_dim)

        if attention_mask is not None:
            att_weights = att_weights.masked_fill(attention_mask == 0, float('-inf'))
            
        att_weights = torch.softmax(
                att_weights, 
                dim=-1
            )

        # print(att_weights)

        att_weights = self.attention_dropout(att_weights)

        # print(att_weights)

        att_results = att_weights @ V
        output =  self.output_proj(att_results)
        
        return output

In [7]:
X = torch.rand(3, 4, 2)
b = torch.tensor(
    [
        [1, 1, 1, 0],
        [1, 1, 0, 0],
        [1, 0, 0, 0],
    ]
)
print(b.shape)
mask = b.unsqueeze(dim=1).repeat(1, 4, 1)

net = SelfAttV4(2)
net(X, mask)

torch.Size([3, 4])


tensor([[[ 0.8148, -1.3758],
         [ 0.8147, -1.3757],
         [ 0.5958, -1.1714],
         [ 0.8135, -1.3742]],

        [[ 0.7603, -1.3603],
         [ 0.7600, -1.3600],
         [ 0.7604, -1.3603],
         [ 0.4107, -1.0053]],

        [[ 0.9528, -1.4962],
         [ 0.9528, -1.4962],
         [ 0.9528, -1.4962],
         [ 0.9528, -1.4962]]], grad_fn=<UnsafeViewBackward0>)

### 5. MultiHeadAttention

In [28]:
class MultiHeadAttention(nn.Module):
    def __init__(self, hidden_dim, num_heads, dropout_rate=0.1):

        super().__init__()

        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads

        # 一般默认有 bias，需要时刻注意，hidden_dim = head_dim * nums_head，所以最终是可以算成是 n 个矩阵
        self.query_proj = nn.Linear(hidden_dim, hidden_dim)
        self.key_proj = nn.Linear(hidden_dim, hidden_dim)
        self.value_proj = nn.Linear(hidden_dim, hidden_dim)

        # gpt2 和 bert 类都有，但是 llama 其实没有
        self.attention_dropout = nn.Dropout(dropout_rate)
        
        # 输出时候的 proj
        self.output_proj = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x, attention_mask=None):
        
        # 需要在 mask 之前 masked_fill
        # X shape is (batch, seq, hidden_dim)
        batch_size, seq_len, _ = x.size()
        
        # shape 变成 （batch_size, num_head, seq_len, head_dim）
        Q = self.query_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.key_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.value_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        print('Q shape:', Q.shape)

        # 注意：这里需要用 head_dim，而不是 hidden_dim
        att_weights = Q @ K.transpose(-1, -2) / math.sqrt(self.head_dim) # (batch, num_head, seq_len, seq_len)
        print(type(attention_mask))

        if attention_mask is not None:
            att_weights = att_weights.masked_fill(
                attention_mask == 0, float('-inf')
                )

        print('att_weights shape:', att_weights.shape)

        # 第四个维度 softmax
        att_weights = torch.softmax(att_weights, dim=-1)
        att_weights = self.attention_dropout(att_weights)

        print('after softmax att_weights shape:', att_weights.shape)

        # (batch, num_head, seq_len, head_dim) 
        output_mid = att_weights @ V
        # att_weights: [B, H, S, S]
        # V:           [B, H, S, D]

        print('first output_mid shape:', output_mid.shape)
        # 重新变成 (batch, seq_len, num_head, head_dim)
        # 这里的 contiguous() 是相当于返回一个连续内存的 tensor，一般用了 permute/tranpose 都要这么操作
        # 如果后面用 Reshape 就可以不用这个 contiguous()，因为 view 只能在连续内存中操作
        output_mid = output_mid.transpose(1, 2).contiguous() # [B, S, H, D]
        output_mid = output_mid.view(batch_size, seq_len, -1)

        print('final output_mid shape:', output_mid.shape)

        output = self.output_proj(output_mid)   # [B, S, hidden_dim]
        
        return output

In [29]:
attention_mask = (
    torch.tensor(
        [
            [0, 1],
            [0, 0],
            [1, 0],
        ]
    )
    .unsqueeze(1)
    .unsqueeze(2)
    .expand(3, 8, 2, 2)
)
x = torch.rand(3, 2, 128)
net = MultiHeadAttention(128, 8)
net(x, attention_mask).shape

Q shape: torch.Size([3, 8, 2, 16])
<class 'torch.Tensor'>
att_weights shape: torch.Size([3, 8, 2, 2])
after softmax att_weights shape: torch.Size([3, 8, 2, 2])
first output_mid shape: torch.Size([3, 8, 2, 16])
final output_mid shape: torch.Size([3, 2, 128])


torch.Size([3, 2, 128])